# 62 — Train + eval SID generator (W3)

Fine-tunes Qwen2.5-1.5B-Instruct + LoRA to emit 3 SID tokens for each W2 query.
Smoke mode: 200 steps, ~10 min. Full: 3 epochs over ~90K rows, ~3-4 hr on L4 / ~1-2 hr on Blackwell.

**Prereqs**: W1 + W2 artifacts on Drive at `/content/drive/MyDrive/recsys2026/sid/track_to_sid.parquet`
and `/content/drive/MyDrive/recsys2026/sid_training/{train,val}.parquet`. HF token in Colab Secrets as `HF_TOKEN`.

> **W3 v2 (2026-05-17):** This run uses the v2 W1 codebook (39,263 unique SIDs vs v1's 3,017). Hub repo is `qwen15b-v2-merged` so v1 remains available for comparison.


In [ ]:
# 1) GPU check.
!nvidia-smi | head -20

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

In [ ]:
# 3) HF auth — pull HF_TOKEN from Colab Secrets (same pattern as notebook 61).
# Setup: Colab → 🔑 Secrets pane → add `HF_TOKEN` with notebook access enabled.
import os
from google.colab import userdata
from huggingface_hub import login
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
print('HF auth OK')

In [ ]:
# 4) Mount Drive + symlink W1/W2 artifacts. Drive subdir naming matches notebook 61:
#   recsys2026_sid_cache              (W1 quantizer output)
#   recsys2026_sid_training_cache     (W2 generator training data)
#   recsys2026_sid_eval_cache         (W3 eval metrics — created here)
#   recsys2026_sid_generator_cache    (W3 LoRA + merged checkpoints)
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=False)

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
SYMLINKS = [
    ('sid',           f'{DRIVE_BASE}/recsys2026_sid_cache'),
    ('sid_training',  f'{DRIVE_BASE}/recsys2026_sid_training_cache'),
    ('sid_eval',      f'{DRIVE_BASE}/recsys2026_sid_eval_cache'),
    ('sid_generator', f'{DRIVE_BASE}/recsys2026_sid_generator_cache'),
]
for local_name, drive_path in SYMLINKS:
    dst = f'{LOCAL_BASE}/{local_name}'
    os.makedirs(drive_path, exist_ok=True)
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(drive_path, dst)
    print(f'symlink: {dst} -> {drive_path}')

# Verify W1 + W2 artifacts visible. If these fail, notebooks 60/61 didn't ship to Drive.
!ls -la experiments/cache/sid/track_to_sid.parquet
!ls -la experiments/cache/sid_training/train.parquet experiments/cache/sid_training/val.parquet

In [ ]:
# 5) Install/upgrade deps. Colab Pro base ships transformers + datasets + torch but
# their preinstalled torchao (0.10.0) is incompatible with recent peft (which requires
# torchao > 0.16). Upgrade torchao explicitly along with peft. Per project memory
# `feedback_colab_library_traps.md` trap #6.
# tensorboard is for the live training-metrics graphs (cell 11 below).
!pip install -q -U \
    "peft>=0.10" \
    "transformers>=4.40" \
    "accelerate>=0.30" \
    "trl>=0.8" \
    "torchao>=0.17" \
    "tensorboard"
import transformers, peft, torch, torchao
print(f'transformers={transformers.__version__}, peft={peft.__version__}, '
      f'torch={torch.__version__}, torchao={torchao.__version__}')

In [ ]:
# 6) Pytest pre-flight on the SID modules. NO output truncation — if anything fails,
# we need to see the full traceback (--tb=long) and ALL collection errors.
!cd /content/recsys2026 && python -m pytest \
    tests/test_sid_vocab.py \
    tests/test_sid_training_format.py \
    tests/test_sid_inference.py \
    tests/test_sid_eval.py \
    -v --tb=long --no-header 2>&1

In [ ]:
# 7) FULL training run — Qwen2.5-1.5B + LoRA + modules_to_save, 3 epochs.
# Wallclock: ~3-4 hr on L4 / ~1-2 hr on Blackwell.
#
# Storage strategy (per discussion):
#   - output_dir on EPHEMERAL Colab disk (NOT Drive) → trainer writes ~975 MB/checkpoint
#     (with --save-best, optimizer state dropped → 5x smaller than default)
#   - --save-best: keep only best+latest checkpoint by val loss (~1.95 GB peak vs ~9.6 GB).
#     Trade-off: NO RESUME (no optimizer state). On L4 (3-4 hr run + Colab disconnects),
#     consider DROPPING --save-best for resumability (default uses ~9.6 GB checkpoints with
#     optimizer state but supports --resume after a crash). On Blackwell (~1-2 hr) the
#     disconnect risk is lower so --save-best is safer.
#   - --merge: push merged 3 GB model to Hub at end
#   - --cleanup-after-push: delete local copies once Hub push succeeds
#   - --results-dir → Drive: persists tensorboard runs/ + results.txt past Colab death (~5 MB)
#
# Net Drive footprint: ~5 MB (logs only). HF Hub footprint: ~4 GB (LoRA + merged).
import os
RESULTS_DIR = '/content/drive/MyDrive/recsys2026_sid_generator_cache/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
LOG_PATH = '/content/drive/MyDrive/recsys2026_sid_generator_cache/training_log_full.txt'

# Speedup config tested on Blackwell 97GB: batch 16 × grad-accum 2 = effective 32
# (matches original spec so no LR adjustment) + no gradient checkpointing.
# Note: --micro-batch 32 with no-checkpointing OOMed even on Blackwell at max_prompt_len=1024,
# so 16 is the safe sweet spot. Wallclock ~1-1.5 hr (vs 3-4 hr with defaults).
# Add --epochs 2 to halve to ~45 min if you're willing to risk under-training.
# On L4 24GB: REMOVE --no-gradient-checkpointing (will OOM).
!cd /content/recsys2026 && python -u scripts/train_sid_generator.py \
    --output-dir /content/recsys2026_full_run \
    --hub-repo OrRim123/recsys2026-sid-generator-qwen15b-v2 \
    --merge \
    --save-best \
    --cleanup-after-push \
    --results-dir {RESULTS_DIR} \
    --micro-batch 16 \
    --grad-accum 2 \
    --epochs 2 \
    --no-gradient-checkpointing \
    2>&1 | tee {LOG_PATH}

In [ ]:
# 8) EVAL — constrained-beam decode over the val parquet (raw slice = matches Blind-A).
# Uses the FULL merged model from cell 7. ~15-25 min for ~760 val rows on L4.
!cd /content/recsys2026 && python -u scripts/eval_sid_generator.py \
    --model-id OrRim123/recsys2026-sid-generator-qwen15b-v2-merged \
    --eval-slice raw \
    2>&1 | tee /content/drive/MyDrive/recsys2026_sid_generator_cache/eval_log_full.txt

In [ ]:
# 9) Read + display final gate metrics.
import json
m = json.load(open('experiments/cache/sid_eval/w3_eval_metrics.json'))
print(json.dumps(m, indent=2))
print()
print('=' * 60)
if m.get('gate_pass'):
    print(f"GATE PASS — mean nDCG@20={m['mean_ndcg_at_20']:.4f} "
          f"(threshold 0.12; delta vs Phase 0 = {m['delta_vs_phase0']:+.4f})")
else:
    print(f"GATE FAIL — mean nDCG@20={m['mean_ndcg_at_20']:.4f} (threshold 0.12)")
    if m.get('paired_bootstrap_ci'):
        ci = m['paired_bootstrap_ci']
        print(f"  paired-bootstrap CI = ({ci['lo']:.4f}, {ci['hi']:.4f})")
print('=' * 60)


## After the run

**Gate pass** (nDCG@20 ≥ 0.12 AND CI lower-bound > 0):
- Merged model is on Hub at `OrRim123/recsys2026-sid-generator-qwen15b-v2-merged`
- Proceed to W4: build `SID_GENERATOR` retrieval class + register `wrrf_bm25_dense_sid_v1`
- Update `MEMORY.md` with the W3 result file

**Gate fail**:
- If point-estimate is close (0.10-0.12) but CI includes 0: more training (5 epochs), check loss curve
- If point-estimate is low (<0.08): likely the W1 SID coarseness biting (3017 unique SIDs limits ceiling).
  Re-run W1 with smaller latent_dim (256→128) + larger codebook (256→512), then re-run W2 + W3.
- If loss diverged: drop LR to 1e-4, re-run.


In [ ]:
# 10) TensorBoard — live training-metrics graphs (loss, grad_norm, lr, eval_loss).
# Single magic that loads BOTH the live in-progress run AND archived past runs:
#   - live: /content/recsys2026_full_run/runs (deleted at end of training by --cleanup-after-push)
#   - archive: /content/drive/MyDrive/recsys2026_sid_generator_cache/results (per-run dirs persist)
# Run this cell ONCE after cell 7 STARTS — the dashboard updates in real time.
# Post-cleanup the live tab will be empty but archive tab keeps every prior run for comparison.
%load_ext tensorboard
%tensorboard --logdir_spec live:/content/recsys2026_full_run/runs,archive:/content/drive/MyDrive/recsys2026_sid_generator_cache/results

## 11) Post-mortem diagnostic — Q1/Q2/Q3 (run on demand)

Use this when the gate (cell 9) failed and you need to know **why** a clean training
run produced a low nDCG@20. Three checks, single inference pass over the val parquet:

- **Q1** — per-position teacher-forced top-k accuracy: which of the 3 SID positions
  is the model wrong on?
- **Q2** — generated-SID popularity stats: is the model collapsing to a few popular
  SIDs regardless of query?
- **Q3** — unconstrained validity: did the model learn the SID space, or only emit
  valid SIDs because of the trie?

Loads the merged model from Hub — does **not** require having just trained.
Wallclock: ~15-25 min on L4 (~760 raw val rows; ~1-1.5 sec/row × 3 forward passes).


In [ ]:
# 11a) Run the post-mortem diagnostic. Reads val.parquet + track_to_sid.parquet
# (already symlinked from Drive in cell 4). Writes the JSON summary to
# experiments/cache/sid_eval/diagnostic_v1/sid_diagnostic_metrics.json.
#
# --limit 0 = full val set; set --limit 200 for a quick smoke first.
!cd /content/recsys2026 && python -u scripts/diagnose_sid_generator.py \
    --model-id OrRim123/recsys2026-sid-generator-qwen15b-v2-merged \
    --eval-slice raw \
    --limit 0 \
    --output-dir experiments/cache/sid_eval/diagnostic_v1 \
    2>&1 | tee /content/drive/MyDrive/recsys2026_sid_generator_cache/diagnostic_log_full.txt


In [ ]:
# 11b) Read + pretty-print the diagnostic. Each section gets a one-line
# interpretation so you don't have to memorize the thresholds.
import json
from pathlib import Path

PATH = Path('experiments/cache/sid_eval/diagnostic_v1/sid_diagnostic_metrics.json')
m = json.loads(PATH.read_text())

print('=' * 70)
print(f"model:    {m['model_id']}")
print(f"queries:  {m['n_queries']}  (slice={m['eval_slice']})")
print('=' * 70)

# ---- Q1: per-position accuracy ----
if 'q1_per_position_accuracy' in m:
    q1 = m['q1_per_position_accuracy']
    print('\n[Q1] Per-position teacher-forced accuracy')
    print(f"{'level':<8}{'top-1 (in-level)':>22}{'top-5 (in-level)':>22}{'unconstr. argmax in level':>30}")
    for lvl in (0, 1, 2):
        s = q1[f'level_{lvl}']
        print(f"{lvl:<8}{s['top_1_restricted_acc']:>22.4f}{s['top_5_restricted_acc']:>22.4f}{s['unrestricted_top1_in_level_rate']:>30.4f}")
    # Interpretation
    accs = [q1[f'level_{l}']['top_1_restricted_acc'] for l in (0,1,2)]
    print(f"\n  -> position-0 top-1 = {accs[0]:.3f}; if << random (1/256 = 0.004) the model "
          f"isn't learning the FIRST code at all.")
    if accs[1] < accs[0] * 0.5 or accs[2] < accs[0] * 0.5:
        print('  -> cascading failure: later positions much worse than position 0.')
    elif max(accs) < 0.05:
        print('  -> all positions near-random — model output is essentially decoupled from query.')

# ---- Q2: popularity collapse ----
if 'q2_popularity_collapse' in m:
    q2 = m['q2_popularity_collapse']
    g, gold = q2['generated'], q2['gold']
    print('\n[Q2] Popularity collapse (generated vs gold distribution)')
    print(f"{'metric':<28}{'generated':>18}{'gold':>18}")
    for label, key, fmt in [
        ('n unique SIDs',           'n_unique',     '{:>18d}'),
        ('top-1 SID frequency',     'top_1_frequency', '{:>18.4f}'),
        ('top-10 coverage',         None,           '{:>18.4f}'),
        ('top-100 coverage',        None,           '{:>18.4f}'),
        ('entropy (bits)',          'entropy_bits', '{:>18.3f}'),
        ('Gini coefficient',        'gini',         '{:>18.3f}'),
    ]:
        if label == 'top-10 coverage':
            gv, av = g['top_k_coverage'].get('10', g['top_k_coverage'].get(10, 0)), gold['top_k_coverage'].get('10', gold['top_k_coverage'].get(10, 0))
        elif label == 'top-100 coverage':
            gv, av = g['top_k_coverage'].get('100', g['top_k_coverage'].get(100, 0)), gold['top_k_coverage'].get('100', gold['top_k_coverage'].get(100, 0))
        else:
            gv, av = g[key], gold[key]
        print(f"{label:<28}{fmt.format(gv):>18}{fmt.format(av):>18}")
    print('\n  top-5 GENERATED SIDs:', q2['top_5_generated_sids'])
    print('  top-5 GOLD SIDs:     ', q2['top_5_gold_sids'])
    if g['top_1_frequency'] > 3 * gold['top_1_frequency']:
        print(f"\n  -> COLLAPSE: top-1 generated SID = {g['top_1_frequency']:.1%} of queries "
              f"(gold = {gold['top_1_frequency']:.1%}). Model is regurgitating popular SIDs.")
    elif g['n_unique'] < gold['n_unique'] / 5:
        print(f"\n  -> diversity collapse: only {g['n_unique']} unique SIDs generated vs "
              f"{gold['n_unique']} gold.")

# ---- Q3: unconstrained validity ----
if 'q3_unconstrained_validity' in m:
    q3 = m['q3_unconstrained_validity']
    print('\n[Q3] Unconstrained-generation validity (no trie)')
    print(f"  position 0 in level-0 range:  {q3['level_0_in_range_rate']:.4f}")
    print(f"  position 1 in level-1 range:  {q3['level_1_in_range_rate']:.4f}")
    print(f"  position 2 in level-2 range:  {q3['level_2_in_range_rate']:.4f}")
    print(f"  all three positions valid:    {q3['all_levels_in_range_rate']:.4f}")
    print(f"  full triplet in codebook:     {q3['triplet_in_codebook_rate']:.4f}")
    if q3['all_levels_in_range_rate'] < 0.5:
        print('\n  -> model has NOT learned the SID grammar — it only stays valid because of the trie.')
    elif q3['triplet_in_codebook_rate'] < 0.3:
        print('\n  -> model knows the grammar but emits triplets not in the codebook ' +
              f"({q3['triplet_in_codebook_rate']:.1%} valid). Generation is mostly hallucinated SIDs.")
    else:
        print(f"\n  -> {q3['triplet_in_codebook_rate']:.1%} of unconstrained generations land on a real SID. "
              'Grammar + codebook are learned; failure mode is fine-grained code selection.')

print('\n' + '=' * 70)
print('Full per-query JSONL at: experiments/cache/sid_eval/diagnostic_v1/per_query.jsonl')
